---
## 1️⃣ Temel Kavramlar

### Lambda Nedir?
```
Sunucu yönetmeden kod çalıştırma (Serverless)

Geleneksel:                    Lambda:
┌─────────────────┐            ┌─────────────────┐
│ Sunucu Kurulumu │            │                 │
│ OS Yönetimi     │     →      │  Sadece Kod     │
│ Güvenlik        │            │  Yaz & Çalıştır │
│ Ölçeklendirme   │            │                 │
└─────────────────┘            └─────────────────┘
```

### API Gateway Nedir?
```
İnternet → API Gateway → Lambda → Cevap

• HTTP isteklerini alır
• Lambda'yı tetikler
• Cevabı döner
```

### SAM CLI Nedir?
```
Serverless Application Model

• template.yaml → Altyapıyı tanımlar
• sam build    → Kodu paketler
• sam deploy   → AWS'e gönderir
• sam local    → Lokal test (Docker ile)
```

---
## 2️⃣ SAM Template Yapısı (template.yaml)

```yaml
# Her SAM template bu header ile başlar
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31
Description: Uygulama açıklaması

# Global ayarlar (tüm fonksiyonlar için)
Globals:
  Function:
    Timeout: 30         # Maksimum çalışma süresi (saniye)
    MemorySize: 128     # RAM (MB) - 128-10240 arası
    Runtime: python3.9  # Çalışma ortamı

# Kaynaklar (Lambda, API Gateway, vs.)
Resources:
  MyFunction:                        # Kaynak adı (sen belirlersin)
    Type: AWS::Serverless::Function  # Kaynak tipi
    Properties:
      Handler: app.handler           # dosya.fonksiyon
      CodeUri: ./                    # Kod dizini
      Events:                        # Tetikleyiciler
        ApiEvent:
          Type: Api
          Properties:
            Path: /hello
            Method: get

# Çıktılar (deploy sonrası gösterilir)
Outputs:
  ApiUrl:
    Value: !Sub https://${ServerlessRestApi}.execute-api.${AWS::Region}.amazonaws.com/Prod/
```

---
## 3️⃣ Lambda Handler Yapısı (app.py)

```python
# En basit Lambda handler
def handler(event, context):
    """
    Lambda fonksiyonu.
    
    Args:
        event: İstek verisi (dict)
            - queryStringParameters: ?key=value
            - body: POST body
            - pathParameters: /users/{id}
            - headers: HTTP headers
        
        context: Lambda context
            - function_name: Fonksiyon adı
            - memory_limit_in_mb: RAM limiti
            - remaining_time_in_millis(): Kalan süre
    
    Returns:
        dict: API Gateway response formatı
    """
    return {
        "statusCode": 200,
        "headers": {
            "Content-Type": "application/json"
        },
        "body": '{"message": "Hello World!"}'
    }
```

---
## 4️⃣ SAM CLI Komutları

```bash
# 1. Proje oluştur (opsiyonel)
sam init

# 2. Build et (deployment paketi oluştur)
sam build
# Çıktı: .aws-sam/build/ dizini

# 3. Deploy et
sam deploy --guided          # İlk kez (interaktif)
sam deploy --resolve-s3      # Sonraki (otomatik S3 bucket)

# 4. Lokal test (Docker gerekli)
sam local start-api          # API Gateway simülasyonu
sam local invoke MyFunction  # Tek fonksiyon çağır

# 5. Logları gör
sam logs -n MyFunction --tail

# 6. Stack sil
sam delete
```

---
## 5️⃣ Event Yapısı (API Gateway → Lambda)

```python
# GET /search?lat=40.96&lon=29.06 isteği geldiğinde:
event = {
    "httpMethod": "GET",
    "path": "/search",
    "queryStringParameters": {
        "lat": "40.96",
        "lon": "29.06"
    },
    "headers": {
        "Host": "xxx.execute-api.eu-central-1.amazonaws.com",
        "User-Agent": "curl/7.64.1"
    },
    "body": None,
    "isBase64Encoded": False
}

# POST /data body ile:
event = {
    "httpMethod": "POST",
    "path": "/data",
    "body": '{"name": "test"}',  # JSON string!
    "headers": {
        "Content-Type": "application/json"
    }
}
```

---
## 6️⃣ Response Yapısı (Lambda → API Gateway)

```python
# Başarılı response
return {
    "statusCode": 200,
    "headers": {
        "Content-Type": "application/json",
        "Access-Control-Allow-Origin": "*"  # CORS
    },
    "body": json.dumps({"data": "value"})  # String olmalı!
}

# Hata response
return {
    "statusCode": 400,
    "body": json.dumps({"error": "Bad Request"})
}

# ⚠️ DİKKAT: body her zaman STRING olmalı!
# Yanlış: "body": {"data": "value"}
# Doğru:  "body": '{"data": "value"}'
```

---
## 7️⃣ FastAPI + Mangum (Önerilen Yöntem)

Normal Lambda handler yazmak yerine FastAPI kullanabilirsin:

```python
# app.py
from fastapi import FastAPI
from mangum import Mangum

app = FastAPI()

@app.get("/hello")
def hello():
    return {"message": "Hello World!"}

@app.get("/search")
def search(lat: float, lon: float):
    return {"lat": lat, "lon": lon}

# Lambda handler
handler = Mangum(app)
```

**Avantajları:**
- Otomatik validation
- Swagger docs (/docs)
- Type hints
- Lokal test kolay (uvicorn ile)

---
## 8️⃣ Pratik: AWS'e Bağlan ve Mevcut Lambda'ları Listele

In [ ]:
import sys
sys.path.insert(0, '..')

from config import get_lambda_client, validate_credentials, AWS_REGION

validate_credentials()

In [ ]:
# Mevcut Lambda fonksiyonlarını listele
lambda_client = get_lambda_client()

response = lambda_client.list_functions()

print("⚡ Mevcut Lambda Fonksiyonları:")
print("=" * 60)

for func in response['Functions']:
    print(f"\n📦 {func['FunctionName']}")
    print(f"   Runtime: {func['Runtime']}")
    print(f"   Memory: {func['MemorySize']} MB")
    print(f"   Timeout: {func['Timeout']} s")
    print(f"   Last Modified: {func['LastModified']}")

In [ ]:
# Lambda'yı invoke et (test)
import json

def invoke_lambda(function_name: str, payload: dict = None):
    """
    Lambda fonksiyonunu çağır.
    
    Args:
        function_name: Lambda fonksiyon adı
        payload: Gönderilecek veri
    
    Returns:
        dict: Lambda response
    """
    lambda_client = get_lambda_client()
    
    response = lambda_client.invoke(
        FunctionName=function_name,
        InvocationType='RequestResponse',
        Payload=json.dumps(payload or {})
    )
    
    result = json.loads(response['Payload'].read())
    
    print(f"⚡ Lambda Response:")
    print(f"   Status: {response['StatusCode']}")
    print(f"   Result: {json.dumps(result, indent=2)}")
    
    return result

# Örnek: Mevcut polygon-hunter-dev Lambda'yı çağır
# invoke_lambda("polygon-hunter-dev", {"httpMethod": "GET", "path": "/health"})

---
## 🎯 ALIŞTIRMA: Hello World Lambda Yaz!

Şimdi sıra sende! `hello_world/` klasöründe:

1. **app.py** - Lambda handler
2. **template.yaml** - SAM template
3. **requirements.txt** - Dependencies

### Hedef:
```bash
curl https://xxx.execute-api.eu-central-1.amazonaws.com/dev/hello
# {"message": "Hello World!"}

curl https://xxx.execute-api.eu-central-1.amazonaws.com/dev/hello?name=Kaan
# {"message": "Hello Kaan!"}
```

### app.py İpuçları:

```python
import json

def handler(event, context):
    # 1. Query parametrelerini al
    params = event.get('queryStringParameters') or {}
    name = params.get('name', 'World')
    
    # 2. Response oluştur
    body = {
        "message": f"Hello {name}!"
    }
    
    # 3. API Gateway formatında döndür
    return {
        "statusCode": ???,
        "headers": {"Content-Type": "???"},
        "body": ???  # json.dumps kullan!
    }
```

### template.yaml İpuçları:

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31
Description: Hello World Lambda

Globals:
  Function:
    Timeout: ???      # Kaç saniye?
    MemorySize: ???   # Kaç MB? (128 yeterli)
    Runtime: ???      # python3.9

Resources:
  HelloWorldFunction:
    Type: AWS::Serverless::Function
    Properties:
      Handler: ???    # dosya.fonksiyon (app.handler)
      CodeUri: ./
      Events:
        HelloApi:
          Type: Api
          Properties:
            Path: ???    # /hello
            Method: ???  # get

Outputs:
  ApiUrl:
    Value: !Sub https://${ServerlessRestApi}.execute-api.${AWS::Region}.amazonaws.com/Prod/hello
```

### Deploy Adımları:

```bash
# 1. Klasöre git
cd aws_test_deploy/lambda_api_gateway/hello_world

# 2. .env'i export et (credentials için)
export $(grep -v '^#' ../../../.env | grep -v '^$' | xargs)

# 3. Build et
sam build

# 4. Deploy et
sam deploy --resolve-s3 --stack-name hello-world-api --capabilities CAPABILITY_IAM

# 5. Test et
curl <API_URL>/hello
curl <API_URL>/hello?name=Kaan
```

---
## 🎯 ALIŞTIRMA 2: S3 Okuyan Lambda Yaz!

Hello World'den sonra `s3_reader/` klasöründe:

### Hedef:
```bash
curl https://xxx/read?key=index/quadkey_index.csv
# S3'teki dosyanın içeriğini döndür (ilk 1000 karakter)
```

### İpuçları:

```python
import boto3
import os

s3 = boto3.client('s3')
BUCKET = os.environ.get('S3_BUCKET', 'polygons-hunter-data')

def handler(event, context):
    params = event.get('queryStringParameters') or {}
    key = params.get('key')
    
    if not key:
        return {"statusCode": 400, "body": "key parametresi gerekli"}
    
    # S3'ten oku
    response = s3.get_object(Bucket=BUCKET, Key=key)
    content = response['Body'].read().decode('utf-8')[:1000]
    
    return {
        "statusCode": 200,
        "body": content
    }
```

**template.yaml'da S3 izni ekle:**
```yaml
Properties:
  ...
  Policies:
    - S3ReadPolicy:
        BucketName: polygons-hunter-data
  Environment:
    Variables:
      S3_BUCKET: polygons-hunter-data
```

---
## ✅ Özet Komutlar

```bash
# Build
sam build

# Deploy
sam deploy --resolve-s3 --stack-name <isim> --capabilities CAPABILITY_IAM

# Stack sil
sam delete --stack-name <isim>

# Logları gör
sam logs -n <FunctionName> --stack-name <isim> --tail
```